In [ ]:
!pip install rank_bm25

In [ ]:
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
documents = [
    "This is a list which containing sample documents.",
    "Keywords are important for keyword-based search.",
    "Document analysis involves extracting keywords.",
    "Keyword-based search relies on sparse embeddings.",
    "Understanding document structure aids in keyword extraction.",
    "Efficient keyword extraction enhances search accuracy.",
    "Semantic similarity improves document retrieval performance.",
    "Machine learning algorithms can optimize keyword extraction methods."
]

In [ ]:
# Load pre-trained Sentence Transformer model
model_name = 'sentence-transformers/paraphrase-xlm-r-multilingual-v1'
model = SentenceTransformer(model_name)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-xlm-r-multilingual-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
doc_embeddings = model.encode(documents)
print((len(doc_embeddings), len(doc_embeddings[0])))

(8, 768)


In [ ]:
query = "Natural language processing techniques enhance keyword extraction efficiency."
query_embedding = model.encode(query)

In [ ]:
similarities = cosine_similarity([query_embedding], doc_embeddings)
similarities

array([[0.16948146, 0.4580227 , 0.5675694 , 0.44123292, 0.6316117 ,
        0.75214124, 0.550352  , 0.7448166 ]], dtype=float32)

In [ ]:
documents[np.argmax(similarities)]

'Efficient keyword extraction enhances search accuracy.'

In [ ]:
sorted_indices = np.argsort(similarities[0])[::-1]
sorted_indices

array([5, 7, 4, 2, 6, 1, 3, 0])

In [ ]:
ranked_docs = [(documents[i], similarities[0][i]) for i in sorted_indices]
ranked_docs

[('Efficient keyword extraction enhances search accuracy.',
  np.float32(0.75214124)),
 ('Machine learning algorithms can optimize keyword extraction methods.',
  np.float32(0.7448166)),
 ('Understanding document structure aids in keyword extraction.',
  np.float32(0.6316117)),
 ('Document analysis involves extracting keywords.', np.float32(0.5675694)),
 ('Semantic similarity improves document retrieval performance.',
  np.float32(0.550352)),
 ('Keywords are important for keyword-based search.', np.float32(0.4580227)),
 ('Keyword-based search relies on sparse embeddings.', np.float32(0.44123292)),
 ('This is a list which containing sample documents.', np.float32(0.16948146))]

In [ ]:
print("Ranked Documents:")
for rank, (document, similarity) in enumerate(ranked_docs, start=1):
    print(f"Rank {rank}: Document - '{document}', Similarity Score - {similarity}")

Ranked Documents:
Rank 1: Document - 'Efficient keyword extraction enhances search accuracy.', Similarity Score - 0.7521412372589111
Rank 2: Document - 'Machine learning algorithms can optimize keyword extraction methods.', Similarity Score - 0.7448166012763977
Rank 3: Document - 'Understanding document structure aids in keyword extraction.', Similarity Score - 0.631611704826355
Rank 4: Document - 'Document analysis involves extracting keywords.', Similarity Score - 0.567569375038147
Rank 5: Document - 'Semantic similarity improves document retrieval performance.', Similarity Score - 0.5503519773483276
Rank 6: Document - 'Keywords are important for keyword-based search.', Similarity Score - 0.45802271366119385
Rank 7: Document - 'Keyword-based search relies on sparse embeddings.', Similarity Score - 0.44123291969299316
Rank 8: Document - 'This is a list which containing sample documents.', Similarity Score - 0.16948145627975464


Until this point, the process is called ranking. Similarity-based retrieval or hybrid retrieval from the knowledge base performs the initial ranking of documents. Reranking is then applied to further reorder the retrieved documents and reduce noise.

In [ ]:
top_4_docs = [doc[0] for doc in ranked_docs[:4]]
top_4_docs

['Efficient keyword extraction enhances search accuracy.',
 'Machine learning algorithms can optimize keyword extraction methods.',
 'Understanding document structure aids in keyword extraction.',
 'Document analysis involves extracting keywords.']

In [ ]:
tokenized_top_4_documents = [doc.split() for doc in top_4_docs]
tokenized_top_4_documents

[['Efficient', 'keyword', 'extraction', 'enhances', 'search', 'accuracy.'],
 ['Machine',
  'learning',
  'algorithms',
  'can',
  'optimize',
  'keyword',
  'extraction',
  'methods.'],
 ['Understanding',
  'document',
  'structure',
  'aids',
  'in',
  'keyword',
  'extraction.'],
 ['Document', 'analysis', 'involves', 'extracting', 'keywords.']]

In [ ]:
tokenized_query = query.split()
tokenized_query

['Natural',
 'language',
 'processing',
 'techniques',
 'enhance',
 'keyword',
 'extraction',
 'efficiency.']

# **BM25 Reranker**

In [ ]:
bm25=BM25Okapi(tokenized_top_4_documents)
bm25

In [ ]:
bm25_scores = bm25.get_scores(tokenized_query)
bm25_scores

array([0.1907998 , 0.16686672, 0.17803252, 0.        ])

In [ ]:
# The first(0) and third(2) document is returned
bm25.get_top_n(n=2, documents=top_4_docs, query=tokenized_query)

['Efficient keyword extraction enhances search accuracy.',
 'Understanding document structure aids in keyword extraction.']

In [ ]:
sorted_indices2 = np.argsort(bm25_scores)[::-1]
sorted_indices2

array([0, 2, 1, 3])

In [ ]:
reranked_docs = [(top_4_docs[i], bm25_scores[i]) for i in sorted_indices2]
reranked_docs

[('Efficient keyword extraction enhances search accuracy.',
  np.float64(0.19079979534096053)),
 ('Understanding document structure aids in keyword extraction.',
  np.float64(0.1780325227902643)),
 ('Machine learning algorithms can optimize keyword extraction methods.',
  np.float64(0.1668667199671815)),
 ('Document analysis involves extracting keywords.', np.float64(0.0))]

# **Cross-Encoder Reranker**

In [ ]:
from sentence_transformers import CrossEncoder

In [ ]:
model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L6-v2")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [ ]:
scores = model.predict([(query, doc) for doc in top_4_docs])
print(scores)

[ 3.1378722   0.84216547 -2.9192996  -2.878191  ]


In [ ]:
scored_docs = zip(scores, top_4_docs)
scored_docs

In [ ]:
reranked_docs_cross_encoder = sorted(scored_docs, reverse=True)
reranked_docs_cross_encoder

[(np.float32(3.1378722),
  'Efficient keyword extraction enhances search accuracy.'),
 (np.float32(0.84216547),
  'Machine learning algorithms can optimize keyword extraction methods.'),
 (np.float32(-2.878191), 'Document analysis involves extracting keywords.'),
 (np.float32(-2.9192996),
  'Understanding document structure aids in keyword extraction.')]

# **Cohere Reranker**

In [ ]:
!pip install langchain_cohere cohere

In [ ]:
from google.colab import userdata
from langchain_cohere import CohereRerank

compressor = CohereRerank(cohere_api_key=userdata.get('CO_API_KEY'), model="rerank-v4.0-pro")

response = compressor.client.rerank(
    model="rerank-v4.0-pro",
    query=query,
    documents=top_4_docs
)

In [ ]:
reranked_docs_cohere = [(top_4_docs[i.index], i.relevance_score) for i in response.results]
reranked_docs_cohere

[('Machine learning algorithms can optimize keyword extraction methods.',
  0.916626),
 ('Efficient keyword extraction enhances search accuracy.', 0.91376984),
 ('Understanding document structure aids in keyword extraction.', 0.8462476),
 ('Document analysis involves extracting keywords.', 0.7810635)]